### Learning drift and volatility (of Ornstein-Uhlenbeck) with neural SDEs (containing two MLPs)

In [1]:
import torch 
import torch.nn as nn
import torchsde
import numpy as np
import tqdm  # just for a nice progress bar

#### 1. NSDE model parameters

In [2]:
batch_size = 128
state_size = 1  # 1D SDE
noise_size = 1  # 1D Brownian motion
hidden_size = 16  # size of hidden layers in our MLPs
t_size = 20  # number of time steps to simulate

# training parameters
steps = 500
lr = 1e-3

#### 2. True data source (Ornstein-Uhlenbeck)
$dX_t = \theta (\mu - X_t) dt + \sigma dW_t$

Some properties of the O-U process:
- The only nontrivial process (up to linear transformations on the space and time variables) that is a Gaussian process, a Markov process, and time-homogeneous
- Mean-reverting (tends to drift towards its mean function)
- Continuous analogue of AR(1)
- Originally designed to model velocity of massive Brownian particles with friction

In [3]:
class TrueSDE(nn.Module):
    # define it as a torch module (inheritance) to easily use torchsde
    
    noise_type = 'diagonal'
    sde_type = 'ito'

    def __init__(self, theta=1.0, mu=0.5, sigma=0.3):  # arbitrary choices
        super().__init__()
        self.register_buffer('theta', torch.tensor([[theta]]))
        self.register_buffer('mu', torch.tensor([[mu]]))
        self.register_buffer('sigma', torch.tensor([[sigma]]))

    # drift: theta*(mu-y)
    def f(self, t, y):
        return self.theta * (self.mu - y)
    
    # diffusion: sigma
    def g(self, t, y):
        # we need to broadcast this to the same shape as y
        return self.sigma.expand(y.size(0), y.size(1))

#### 3. NSDE to be trained

In [4]:
class DriftNet(nn.Module):
    """MLP for drift"""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.Tanh(),
            nn.Linear(hidden_size, state_size)
        )
    def forward(self, t, y):
        # assuming independence with t for simplicity (autonomous drift)
        return self.net(y)
    
class DiffusionNet(nn.Module):
    """MLP for diffusion"""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, state_size * noise_size),
            nn.Sigmoid()  # use sigmoid to keep diffusion bounded and positive
        )
    
    def forward(self, t, y):
        # again ignoring t for simplicity
        # reshape output to (batch, state_size, noise_size)
        return self.net(y)
    
    
class NeuralSDE(nn.Module):
    """Container of learned SDE"""
    noise_type = 'diagonal'
    sde_type = 'ito'

    def __init__(self):
        super().__init__()
        self.f_net = DriftNet()
        self.g_net = DiffusionNet()

    # learned drift
    def f(self, t, y):
        return self.f_net(t, y)
    
    # learned diffusion
    def g(self, t, y):
        return self.g_net(t, y)
    

#### 3.5. MMD loss using RBF kernel
While MSE encourages the NSDE to produce smooth, low-volatility paths to match the distribution's instantaneous values, MMD (a "distance" metric for probability distributions) forces the model to match in distribution itself.

In [ ]:
def gaussian_kernel(x, y, sigma=1.0):
    """Computes the Gaussian (RBF) kernel between two tensor batches."""
    # Expand to create the distance matrix: (batch_x, 1, features) - (1, batch_y, features)
    x_expanded = x.unsqueeze(1)
    y_expanded = y.unsqueeze(0)
    
    # Squared Euclidean distance: ||x - y||^2
    sq_dist = torch.sum((x_expanded - y_expanded) ** 2, dim=2)
    
    # Kernel function: exp(-sq_dist / (2 * sigma^2))
    # We use multiple sigma values (bandwidths) for robustness, common in MMD.
    sigma_list = [0.1, 1.0, 10.0]
    kernel_vals = [torch.exp(-sq_dist / (2 * s**2)) for s in sigma_list]
    
    # Sum the kernels to get the multi-scale kernel
    return sum(kernel_vals)

def mmd_loss(x, y):
    """Computes the MMD between two distributions x and y."""
    
    # Ensure x and y are reshaped to (batch_size, total_features)
    # The paths are (t_size, batch_size, state_size). We flatten features:
    # (batch_size, t_size * state_size)
    x = x.permute(1, 0, 2).flatten(start_dim=1)
    y = y.permute(1, 0, 2).flatten(start_dim=1)
    
    # Calculate kernel matrices
    K_xx = gaussian_kernel(x, x)
    K_yy = gaussian_kernel(y, y)
    K_xy = gaussian_kernel(x, y)
    
    # MMD squared is calculated as:
    # E[k(x,x')] + E[k(y,y')] - 2 * E[k(x,y)]
    mmd_val = K_xx.mean() + K_yy.mean() - 2 * K_xy.mean()
    return mmd_val

#### 4. Helper function to source data

In [6]:
true_sde = TrueSDE()
ts = torch.linspace(0, 1, t_size)
y0 = torch.full((batch_size, state_size), 0.5)  # start all path at 0.5

def get_true_paths_batch():
    """Generates a batch of paths from TrueSDE"""
    with torch.no_grad():
        # sdeint generates trajectories
        true_paths = torchsde.sdeint(true_sde, y0, ts, dt=1e-2)

    return true_paths



#### 5. Training loop (now using MMD)

In [15]:
def main():
    print("---Start Training---")

    # initialize model and optimizer
    neural_sde = NeuralSDE()
    optimizer = torch.optim.Adam(neural_sde.parameters(), lr=lr)

    # NO MORE MSE loss
    # loss_fn = nn.MSELoss()

    pbar = tqdm.tqdm(range(1, steps + 1))
    
    for i in pbar:
        optimizer.zero_grad()

        # get a batch of "real" data
        true_paths = get_true_paths_batch()

        # generate a batch of "fake" data from NeuralSDE
        # starting from the same initial condition
        # sdeint uses adjoint backprop
        pred_paths = torchsde.sdeint(neural_sde, y0, ts, dt = 1e-2)

        loss = mmd_loss(pred_paths, true_paths)

        loss.backward()
        optimizer.step()

        if i % 10 == 0:
            pbar.set_description(f"Step: {i}, Loss: {loss.item():.4f}")
        
    print("---Training Complete---")

    print("\n--- Comparing final paths ---")

    # * New: set a different seed for the final simulation
    torch.manual_seed(99)
    np.random.seed(99)

    with torch.no_grad():
        # * These sdeint calls will now use the new (99) seed's noise
        test_y0 = torch.full((1, state_size), 0.5)
        true_final_path = torchsde.sdeint(true_sde, test_y0, ts, dt=1e-2).squeeze()
        pred_final_path = torchsde.sdeint(neural_sde, test_y0, ts, dt=1e-2).squeeze()
        
    # comparing the two paths
    print(f"True SDE mean at T: {true_final_path[-1].item():.4f}")
    print(f"Neural SDE mean at T: {pred_final_path[-1].item():.4f}")

    # inspect the learned functions themselves
    # check learned diffusion at the starting point y=0.5
    test_y = torch.tensor([[0.5]])
    learned_g = neural_sde.g(0., test_y)
    print(f"\n True diffusion g(0.5): {true_sde.g(0., test_y).item():.4f}")
    print(f"Learned diffusion g(0.5): {learned_g.item():.4f}")


In [16]:
if __name__ == "__main__":
    # set a random seed for reproducibility

    torch.manual_seed(42)
    np.random.seed(42)

main()

---Start Training---


Step: 500, Loss: 0.0210: 100%|██████████| 500/500 [41:55<00:00,  5.03s/it]


---Training Complete---

--- Comparing final paths ---
True SDE mean at T: 0.3581
Neural SDE mean at T: 0.4890

 True diffusion g(0.5): 0.3000
Learned diffusion g(0.5): 0.2460


It seems like despite low loss and a reasonable accuracy achieved in learning the mean, our NSDE is still unable to learn the true diffusion under the current set-up.

Possible causes:

1. Model blindly prioritizes lowering loss and "cheat" by being too smooth.
   
Fix: Change the Loss Function: Switch from a simple MSE on the paths to a distribution matching loss (like a GAN or Maximum Mean Discrepancy, MMD✅). These losses compare the statistical properties of the generated paths, making it harder for the model to "cheat" by being too smooth.

2. Choice of activation function.
   
Fix: Remove Sigmoid from DiffusionNet: The <code>nn.Sigmoid()</code> activation keeps the output bounded between $(0, 1)$. Since the true $\sigma$ is $0.3$, this isn't strictly an issue, but a <code>nn.Softplus()</code> (output: $(0, \infty)$) is often preferred for diffusion networks as it ensures positivity while allowing larger values.

3. Inadequate training.
   
Fix: Increase Training Time: The training might need more than 500 steps, especially with a low learning rate, to fully escape the local minimum that favors smoothness. Increase steps and see if the diffusion value starts climbing.

4. Choice of optimizer.
   
Fix: Try a Different Optimizer: While Adam is standard, exploring different optimizers or adding learning rate scheduling can sometimes help escape smooth local minima.

5. Size of MLPs: increase <code>hidden_size</code> to 32, 64, or bigger.

6. MMD kernel bandwidths: if the list of sigmas are not well-tuned to the scale of your data, the MMD metric might be too flat to provide meaningful gradients.